In [ ]:
from pathlib import Path

import joblib
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams["figure.figsize"] = (8, 5)
sns.set_theme(style="whitegrid")

# docker-compose에서 ./shared가 /workspace/shared로 마운트
BASE_DIR = Path("/workspace") if Path("/workspace").exists() else Path(".").resolve()
SHARED_DIR = BASE_DIR / "shared"

MODEL_PATH = SHARED_DIR / "model.pkl"
TEST_PATH = SHARED_DIR / "test.csv"
RESULT_PATH = SHARED_DIR / "result.csv"

print("MODEL_PATH:", MODEL_PATH)
print("TEST_PATH:", TEST_PATH)
print("RESULT_PATH:", RESULT_PATH)

## 1. 공유 폴더 파일 확인

In [ ]:
required_files = [MODEL_PATH, TEST_PATH]
for path in required_files:
    print(path, "exists:", path.exists())

if not MODEL_PATH.exists() or not TEST_PATH.exists():
    raise FileNotFoundError(
        "shared/model.pkl 또는 shared/test.csv가 없습니다. "
        "먼저 `docker compose run --rm researcher1`을 실행하세요."
    )

## 2. 모델과 테스트 데이터 로드

In [ ]:
# 연구자 1이 저장한 scikit-learn Pipeline 모델을 불러오기
model = joblib.load(MODEL_PATH)

# 연구자 1 컨테이너에서 공유 폴더로 전달된 test.csv
test_df = pd.read_csv(TEST_PATH)

print("test shape:", test_df.shape)
display(test_df.head())
print(model)

## 3. 추론 수행

In [ ]:
predictions = model.predict(test_df)

# 성취도 지수는 10~100 범위 제한
predictions = predictions.clip(10, 100)

result_df = test_df.copy()
result_df["Predicted Performance Index"] = predictions

# 제출/확인용 결과 파일 저장
result_df.to_csv(RESULT_PATH, index=False)

print("result saved:", RESULT_PATH)
display(result_df.head())

## 4. 추론 결과 시각화

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(result_df["Predicted Performance Index"], bins=30, kde=True)
plt.title("Distribution of Predicted Performance Index")
plt.xlabel("Predicted Performance Index")
plt.ylabel("Count")
plt.show()

plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=result_df,
    x="Previous Scores",
    y="Predicted Performance Index",
    alpha=0.5
)
plt.title("Previous Scores vs Predicted Performance Index")
plt.xlabel("Previous Scores")
plt.ylabel("Predicted Performance Index")
plt.show()